# Union Quesions

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

%md
## Que1: Insurance Customer Data Merge

**Difficulty:** Easy

### Problem

An insurance company maintains customer records in two separate regional systems.

Combine all customer records from `ic_data_1` and `ic_data_2` into a single dataset using **UNION ALL**. Do not remove duplicate rows. Return the customer details and sort the combined result by `age` in ascending order.

**Schema columns:**

- **ic_data_1:** `customer_id`, `first_name`, `last_name`, `age`, `policy_type`
- **ic_data_2:** `customer_id`, `first_name`, `last_name`, `age`, `policy_type`

**Output columns:** `customer_id`, `first_name`, `last_name`, `age`, `policy_type`

Order the result by `age` in ascending order.

### Examples

#### Example 1

**Input:**

**ic_data_1:**

| customer_id | first_name | last_name | age | policy_type |
|------------:|------------|-----------|----:|-------------|
| 1 | Alice | Smith | 30 | auto |
| 2 | Bob | Johnson | 40 | home |
| 3 | Carol | Williams | 35 | life |

**ic_data_2:**

| customer_id | first_name | last_name | age | policy_type |
|------------:|------------|-----------|----:|-------------|
| 4 | Dave | Brown | 45 | auto |
| 5 | Eve | Jones | 55 | health |
| 6 | Frank | Davis | 60 | life |

**Output:**

| customer_id | first_name | last_name | age | policy_type |
|------------:|------------|-----------|----:|-------------|
| 1 | Alice | Smith | 30 | auto |
| 3 | Carol | Williams | 35 | life |
| 2 | Bob | Johnson | 40 | home |
| 4 | Dave | Brown | 45 | auto |
| 5 | Eve | Jones | 55 | health |
| 6 | Frank | Davis | 60 | life |

**Explanation:** Combine all rows from both tables using `UNION ALL` without removing duplicates. Finally, sort the merged result by `age` in ascending order.

### Constraints

- Use `UNION ALL` to combine both tables.
- Do not remove duplicate rows.
- Return exactly the required output columns.
- Order the result by `age` in ascending order.

In [0]:
ic_data_1_data=[(1,"Alice","Smith",30,"auto"),(2,"Bob","Johnson",40,"home"),(3,"Carol","Williams",35,"life")]
ic_data_1_df=spark.createDataFrame(ic_data_1_data,["customer_id","first_name","last_name","age","policy_type"])
display(ic_data_1_df)

ic_data_2_data=[(4,"Dave","Brown",45,"auto"),(5,"Eve","Jones",55,"health"),(6,"Frank","Davis",60,"life")]
ic_data_2_df=spark.createDataFrame(ic_data_2_data,["customer_id","first_name","last_name","age","policy_type"])
display(ic_data_2_df)

combined_df = ic_data_1_df.union(ic_data_2_df).orderBy("age")
display(combined_df)



%md
## Que2: Rearrange Products Table

**Difficulty:** Easy

### Problem

A pricing system stores product prices across three different store columns.

Transform the table so that each available store price becomes a separate row. Return the product ID, the corresponding store label (`store1`, `store2`, or `store3`), and its price. Ignore stores where the price is `NULL`.

**Schema columns:** `products.product_id`, `products.store1`, `products.store2`, `products.store3`

**Output columns:** `product_id`, `store`, `price`

Order the result by `product_id` in ascending order, then by `store` in ascending order.

### Examples

#### Example 1

**Input:**

**products:**

| product_id | store1 | store2 | store3 |
|-----------:|-------:|-------:|-------:|
| 1 | 10 | 12 | NULL |
| 2 | NULL | 20 | 18 |
| 3 | 5 | NULL | NULL |

**Output:**

| product_id | store | price |
|-----------:|--------|------:|
| 1 | store1 | 10 |
| 1 | store2 | 12 |
| 2 | store2 | 20 |
| 2 | store3 | 18 |
| 3 | store1 | 5 |

**Explanation:** Each non-null store price is converted into a separate row. Product 1 generates two rows because it has prices in `store1` and `store2`, while its missing `store3` price is ignored.

### Constraints

- Omit rows where the store price is `NULL`.
- Store labels must be `store1`, `store2`, or `store3`.
- Return the output columns in the required order.
- Order the result by `product_id`, then `store`.

In [0]:
products_data=[(1,10,12,None),(2,None,20,18),(3,5,None,None)]
products_df=spark.createDataFrame(products_data,["product_id","store1","store2","store3"])
display(products_df)

store1_df = products_df.withColumn("store", lit("store1")).select("product_id", "store", col("store1").alias("price"))
store2_df = products_df.withColumn("store", lit("store2")).select("product_id", "store", col("store2").alias("price"))
store3_df = products_df.withColumn("store", lit("store3")).select("product_id", "store", col("store3").alias("price"))


compined_df = store1_df.union(store2_df).union(store3_df).dropna(subset=["price"]).orderBy("product_id", "store")
display(compined_df)
